# Inspect Sharded Dataset

This notebook provides tools to inspect a single shard from the processed dataset. It includes:
1. **Data Loading**: Load .pt shard files.
2. **Statistics**: Analyze label distribution and audio feature statistics.
3. **Visualization**: Interactive plots for Log Mel Spectrograms and extracted Video Frames.

In [ ]:
import torch
import io
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import os
from collections import Counter

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 6]

In [ ]:
# --- Configuration ---
SHARD_PATH = "../tests/dummy_shard.pt" # Replace with your actual shard path like '../data/fakeavceleb/preprocessed/shards_train/shard_0000.pt'
NUM_SAMPLES_VIZ = 4
IM_RES = 224

In [ ]:
def load_shard(path):
    if not os.path.exists(path):
        print(f"File not found: {path}")
        return []
    try:
        print(f"Loading shard from {path}...")
        data = torch.load(path, map_location='cpu')
        print(f"Successfully loaded {len(data)} samples.")
        return data
    except Exception as e:
        print(f"Error loading shard: {e}")
        return []

def analyze_shard(data):
    if not data:
        return

    print("\n--- Shard Statistics ---")
    
    # Label Distribution
    labels = [d.get('labels', 'Unknown') for d in data]
    label_counts = Counter(labels)
    print(f"Label Distribution: {dict(label_counts)}")

    # Audio Statistics
    fbank_means = []
    fbank_stds = []
    audio_lengths = []
    
    for x in data:
        fb = x['fbank']
        if isinstance(fb, torch.Tensor):
            fb = fb.float().numpy()
            
        fbank_means.append(fb.mean())
        fbank_stds.append(fb.std())
        audio_lengths.append(fb.shape[0])

    print(f"Audio Length (frames): Min={min(audio_lengths)}, Max={max(audio_lengths)}, Mean={np.mean(audio_lengths):.1f}")
    print(f"Fbank Mean (global approx): {np.mean(fbank_means):.4f}")
    print(f"Fbank Std (global approx): {np.mean(fbank_stds):.4f}")

    # Plots
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # 1. Label Dist
    sns.barplot(x=list(label_counts.keys()), y=list(label_counts.values()), hue=list(label_counts.keys()), ax=axes[0], palette="viridis", legend=False)
    axes[0].set_title("Label Distribution")
    axes[0].set_ylabel("Count")

    # 2. Audio Length Dist
    sns.histplot(audio_lengths, bins=20, ax=axes[1], kde=False, color="blue")
    axes[1].set_title("Audio Length Distribution (Frames)")
    
    # 3. Fbank Value Dist (Mean of each sample)
    sns.histplot(fbank_means, bins=30, ax=axes[2], kde=True, color="green")
    axes[2].set_title("Distribution of Mean Fbank Values")

    plt.tight_layout()
    plt.show()

In [ ]:
def visualize_samples(data, num_samples=4):
    if not data:
        return
        
    samples = random.sample(data, min(num_samples, len(data)))
    
    print(f"\n--- Visualizing {len(samples)} Random Samples ---")
    
    fig, axes = plt.subplots(len(samples), 2, figsize=(20, 5 * len(samples)))
    if len(samples) == 1:
        axes = [axes]
        
    for i, sample in enumerate(samples):
        video_id = sample.get('video_id', 'Unknown ID')
        label = sample.get('labels', '?')
        fbank = sample['fbank']
        images_bytes = sample['images']
        
        # Audio Plot
        ax_audio = axes[i][0]
        if isinstance(fbank, torch.Tensor):
            fbank_np = fbank.float().cpu().t().numpy()
        else:
            fbank_np = fbank.T.astype(np.float32)
            
        im_plot = ax_audio.imshow(fbank_np, aspect='auto', origin='lower', cmap='inferno')
        ax_audio.set_title(f"Spectrogram: {video_id} (Label: {label})")
        ax_audio.set_xlabel("Time")
        ax_audio.set_ylabel("Mel Bin")
        plt.colorbar(im_plot, ax=ax_audio)

        # Video Plot (Contact Sheet)
        ax_video = axes[i][1]
        ax_video.axis('off')
        
        frames = []
        for img_byte in images_bytes:
            img = Image.open(io.BytesIO(img_byte))
            frames.append(np.array(img))
            
        if frames:
            # Create 4x4 grid (assuming 16 frames)
            n = len(frames)
            grid_dim = int(np.ceil(np.sqrt(n)))
            h, w, c = frames[0].shape
            
            canvas = np.zeros((grid_dim * h, grid_dim * w, c), dtype=np.uint8)
            for idx, frame in enumerate(frames):
                r, c_idx = divmod(idx, grid_dim)
                canvas[r*h:(r+1)*h, c_idx*w:(c_idx+1)*w] = frame
                
            ax_video.imshow(canvas)
            ax_video.set_title(f"Extracted Frames ({n})")
        else:
            ax_video.text(0.5, 0.5, "No frames available", ha='center')

    plt.tight_layout()
    plt.show()

In [ ]:
# --- Main Execution ---
data = load_shard(SHARD_PATH)
analyze_shard(data)
visualize_samples(data, NUM_SAMPLES_VIZ)